In [10]:
import torch
import json
from mingpt.model import GPT

In [11]:
config = GPT.get_default_config()
config.model_type = 'gpt-mini'  # match training config
config.vocab_size = 85   # must match training vocab
config.block_size = 128          # match training config

In [12]:
device = 'cpu'
model = GPT(config)
model.load_state_dict(torch.load('./out/model.pt', map_location='cpu'))
model.eval()

number of parameters: 2.71M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(85, 192)
    (wpe): Embedding(128, 192)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm((192,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=192, out_features=576, bias=True)
          (c_proj): Linear(in_features=192, out_features=192, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((192,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): ModuleDict(
          (c_fc): Linear(in_features=192, out_features=768, bias=True)
          (c_proj): Linear(in_features=768, out_features=192, bias=True)
          (act): NewGELU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((192,), eps=1e-05, elementwise_affine=True, bias=True)


In [13]:
def generate(prompt='', num_samples=10, steps=20, do_sample=True):
    # Load the same training data to build the vocabulary
    text = open('input.txt', 'r').read()
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    vocab_size = len(chars)

    # Encode the prompt using character-level encoding
    if prompt == '':
        # Start with the first character in the vocabulary
        x = torch.tensor([[stoi[chars[0]]]], dtype=torch.long)
    else:
        x = torch.tensor([[stoi[s] for s in prompt]], dtype=torch.long)

    # Generate
    y = model.generate(x, steps, temperature=1.0, do_sample=True, top_k=10)[0]

    for i in range(num_samples):
        out = ''.join([itos[int(i)] for i in y])
    
    return out

In [62]:
inference_text = generate(prompt='Pacific ', num_samples=1, steps=200)
print(inference_text)

Pacific war of the
capsize."

"Well it, though, to go on. Shipmate, look. Let us so get up and the
watch, then answer, aye, and so fell surrounded, but we do dat drag to,
the pagan captain, because for the su


In [63]:
with open("hexagrams.json", mode="r", encoding="utf-8") as file:
    data = json.load(file)
to_remove = {'of', 'with', 'a', 'the', 'that', 'and', 'in', 'from', 'when', 'why', 'what', 'who', 'how', 'by', 'for', 'to', 'it', 'its', 'as', 'not', 'without', 'at'}
words = set(inference_text.lower().split())
words -= to_remove

for key in data:
    keywords = set("".join(data[key]['keywords']).split())
    matches = words & keywords
    if len(matches) > 0:
        print(data[key]["name"])
        print(matches)

Sun / The Reduction
{'up'}
